In [2]:
# !pip install duckdb
# !pip install sqlalchemy
# !pip install pyodbc
# !pip install pandas
# !pip install polars


In [3]:
import datetime 
from MODULOS.conexion_sql import get_connection
from MODULOS.conexion_sql import get_sqlalchemy_engine
import duckdb
import pandas as pd
from datetime import datetime
from matplotlib.dates import relativedelta
from pathlib import Path

PERIODO = 202609

hora_inicio = datetime.now()

In [4]:
def SQLServer_to_Parquet(ScriptSQL, ruta_parquet):
    
    conn = get_connection()
    if conn is None:
        print("No se pudo conectar a SQL Server")
        return
    try:
        query = f"""
                {ScriptSQL}
        """
        df = pd.read_sql(query, conn)
        print(f"Registros obtenidos: {len(df):,}")
        duckdb.sql(f"""
            COPY (
                SELECT *
                FROM df
            )
            TO '{ruta_parquet}'
            (
                FORMAT PARQUET,
                COMPRESSION ZSTD
            )
        """)
        print("Parquet generado correctamente")
        print(ruta_parquet)
    except Exception as e:
        print(f"Error: {e}")
    finally:
        conn.close()

In [5]:
anio = int(str(PERIODO)[:4])
mes = int(str(PERIODO)[4:])
# Crear el objeto datetime directamente de forma limpia
fecha_ini = datetime(anio, mes, 1) 
fecha_final = (fecha_ini + relativedelta(months=1) ) #- relativedelta(day=1)
fecha_final = fecha_final - relativedelta(days=1)
print(f"Fecha Inicial: {fecha_ini} | Fecha Final: {fecha_final}")


########################################  RUTAS ARCHIVO PARQUET ########################################
PATH_PRINCIPAL = "D:/Datos"
NOMBRE_TABLA = f"FAC.COSTO_DARWIN_BASE_OPERACION_{PERIODO}"

# Definimos la ruta de la carpeta como un objeto Path para poder usar .mkdir()
DIRECTORIO_PARQUET = Path(f"{PATH_PRINCIPAL}/BACKUP_FAC_COSTOS_POR_TRAYECTO/{PERIODO}")

# Creamos la ruta (carpeta y subcarpetas) si no existe
DIRECTORIO_PARQUET.mkdir(parents=True, exist_ok=True)

# Ruta completa del archivo uniendo el directorio y el nombre del archivo
RUTA_PARQUET = DIRECTORIO_PARQUET / NOMBRE_TABLA
########################################################################################################

print("Directorio creado/verificado:", DIRECTORIO_PARQUET)



Fecha Inicial: 2026-09-01 00:00:00 | Fecha Final: 2026-09-30 00:00:00
Directorio creado/verificado: D:\Datos\BACKUP_FAC_COSTOS_POR_TRAYECTO\202609


In [6]:

PERIODO_INICIAL = [str(fecha_ini).replace('-', '')[:8]]
PERIODO_FINAL = [str(fecha_final).replace('-', '')[:8]]
print(PERIODO_INICIAL,'|',PERIODO_FINAL)

RUTA_PARQUET_final = ""

print(int(PERIODO_FINAL[0]))

for date01 in range(int(PERIODO_INICIAL[0]), int(PERIODO_FINAL[0]),10):  

    RUTA_PARQUET_final = f"{RUTA_PARQUET}_{date01}.parquet"
    
    fecha_inicial_aux = datetime(
            int(str(date01)[:4]), 
            int(str(date01)[4:6]), 
            int(str(date01)[6:8]))
        
    fecha_final_aux = datetime(
        int(str(date01 )[:4]), 
        int(str(date01 )[4:6]), 
        int(str(date01 )[6:8])) + relativedelta(days=10)

    SCRIPT_SQL = ""

    if fecha_final_aux + relativedelta(days=1) > fecha_final:
        SCRIPT_SQL = f"""
        SELECT 
        * 
        FROM [FAC].[COSTO_DARWIN_BASE_OPERACION_MES] WITH(NOLOCK) WHERE PERIODO_EJECUCION = {PERIODO} 
        AND  FECHA_HORA_MOVIMIENTO BETWEEN '{fecha_inicial_aux}' AND '{fecha_final + relativedelta(days=1)}'
        OPTION(RECOMPILE)
        """
        print(SCRIPT_SQL)
    else:
        SCRIPT_SQL = f"""
        SELECT 
        * 
        FROM [FAC].[COSTO_DARWIN_BASE_OPERACION_MES] WITH(NOLOCK) WHERE PERIODO_EJECUCION = {PERIODO} 
        AND  FECHA_HORA_MOVIMIENTO BETWEEN '{fecha_inicial_aux}' AND '{fecha_final_aux}'
        OPTION(RECOMPILE)
        """
        print(SCRIPT_SQL)
    
    SQLServer_to_Parquet(
        SCRIPT_SQL,
        RUTA_PARQUET_final
    )
print(f"Finaliza carga de {NOMBRE_TABLA}: {RUTA_PARQUET}")

 ############################### se deja pendiente mientras sse finaliza el trabajo de darwin 

['20260901'] | ['20260930']
20260930

        SELECT 
        * 
        FROM [FAC].[COSTO_DARWIN_BASE_OPERACION_MES] WITH(NOLOCK) WHERE PERIODO_EJECUCION = 202609 
        AND  FECHA_HORA_MOVIMIENTO BETWEEN '2026-09-01 00:00:00' AND '2026-09-11 00:00:00'
        OPTION(RECOMPILE)
        
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_7280\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 2,741,858
Parquet generado correctamente
D:\Datos\BACKUP_FAC_COSTOS_POR_TRAYECTO\202609\FAC.COSTO_DARWIN_BASE_OPERACION_202609_20260901.parquet

        SELECT 
        * 
        FROM [FAC].[COSTO_DARWIN_BASE_OPERACION_MES] WITH(NOLOCK) WHERE PERIODO_EJECUCION = 202609 
        AND  FECHA_HORA_MOVIMIENTO BETWEEN '2026-09-11 00:00:00' AND '2026-09-21 00:00:00'
        OPTION(RECOMPILE)
        
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_7280\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 2,287,625
Parquet generado correctamente
D:\Datos\BACKUP_FAC_COSTOS_POR_TRAYECTO\202609\FAC.COSTO_DARWIN_BASE_OPERACION_202609_20260911.parquet

        SELECT 
        * 
        FROM [FAC].[COSTO_DARWIN_BASE_OPERACION_MES] WITH(NOLOCK) WHERE PERIODO_EJECUCION = 202609 
        AND  FECHA_HORA_MOVIMIENTO BETWEEN '2026-09-21 00:00:00' AND '2026-10-01 00:00:00'
        OPTION(RECOMPILE)
        
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_7280\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 2,558,401
Parquet generado correctamente
D:\Datos\BACKUP_FAC_COSTOS_POR_TRAYECTO\202609\FAC.COSTO_DARWIN_BASE_OPERACION_202609_20260921.parquet
Finaliza carga de FAC.COSTO_DARWIN_BASE_OPERACION_202609: D:\Datos\BACKUP_FAC_COSTOS_POR_TRAYECTO\202609\FAC.COSTO_DARWIN_BASE_OPERACION_202609


In [7]:
df_resultado = duckdb.sql(f"""
    SELECT *
    FROM '{RUTA_PARQUET_final}'
    LIMIT 5
""").df()
display(df_resultado)

,PERIODO_EJECUCION,NUMERO_GUIA,CANAL,NOMBRE_PRODUCTO,MEDIO_DE_PAGO,VALOR_RECAUDO,VALOR_ENVIO,NOMBRE_ESTADOENVIO,ID_PROCESO,NOMBRE_PROCESO_LOG,...,TOTAL_PIEZAS,VALOR_COMISION_RECAUDO_EFECTY,VALOR_COMISION_VENTA_CDS,VALOR_LIQUIDADO,APLICA_EPS_EZU,APLICA_RE_EV,VLT_RURAL_ORIG,VLT_RURAL_DEST,APLICA_CONDICIONAL,CONDICION_ENTRADA_ENTREGADO
0,202609,2.297816e+09,CORPORATIVO,MERCANCIA PREMIER,NO DISPONIBLE,0,40424,ENTREGADO,12,SALIDA A REGIONAL,...,647,0.0,0.0,40424.0,1,1,0,0,LIQUIDADO - NO FACTURADO Y MOVILIZADO,03 - TIENE ENTRADA - FUE ENTREGADO - NO FACTURADO
1,202609,2.297816e+09,CORPORATIVO,MERCANCIA PREMIER,NO DISPONIBLE,0,40424,ENTREGADO,12,SALIDA A REGIONAL,...,405,0.0,0.0,40424.0,1,1,0,0,LIQUIDADO - NO FACTURADO Y MOVILIZADO,03 - TIENE ENTRADA - FUE ENTREGADO - NO FACTURADO
2,202609,2.297816e+09,CORPORATIVO,MERCANCIA PREMIER,NO DISPONIBLE,0,40424,ENTREGADO,6,ENTRADA DE ZONA URBANA,...,61,0.0,0.0,40424.0,1,1,0,0,LIQUIDADO - NO FACTURADO Y MOVILIZADO,03 - TIENE ENTRADA - FUE ENTREGADO - NO FACTURADO
3,202609,9.193070e+09,RETAIL,DOCUMENTO UNITARIO,NO DISPONIBLE,0,8600,ENTREGADO A REMITENTE,24,REPORTADO ENTREGADO,...,41,0.0,0.0,8600.0,1,1,0,0,LIQUIDADO-FACTURADO Y MOVILIZADO,01 - TIENE ENTRADA - FUE ENTREGADO - FACTURADO...
4,202609,2.291266e+09,CORPORATIVO,DOCUMENTO UNITARIO,NO DISPONIBLE,0,5820,ENTREGADO,24,REPORTADO ENTREGADO,...,69,0.0,0.0,5820.0,1,1,0,0,LIQUIDADO - NO FACTURADO Y MOVILIZADO,03 - TIENE ENTRADA - FUE ENTREGADO - NO FACTURADO


In [8]:
hora_final = datetime.now()
duracion = hora_final - hora_inicio
print(f"Duración de la ejecución: {duracion}")

Duración de la ejecución: 2:29:45.048136
